## make a new column pdf id in a new csv

In [1]:
import pandas as pd
import os

# Define file paths
input_csv_path = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions.csv'
output_csv_path = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID.csv'

# Load the original CSV file
df = pd.read_csv(input_csv_path)

# Check the number of columns and rows in the original CSV
original_columns = df.shape[1]
original_rows = df.shape[0]

# Extract PDF_ID from UPLOADED_ANS column
df['PDF_ID'] = df['UPLOADED_ANS'].apply(lambda x: x.split('pdf/')[1] if isinstance(x, str) else None)

# Check the number of columns and rows in the new CSV
new_columns = df.shape[1]
new_rows = df.shape[0]

# Save the new DataFrame to CSV
df.to_csv(output_csv_path, index=False)

# Print details
print(f"Original CSV: {original_columns} columns, {original_rows} rows")
print(f"New CSV: {new_columns} columns, {new_rows} rows")
print(f"New CSV saved at: {output_csv_path}")


/var/folders/62/cwfgmpjx2rq78q__0nz165zh0000gp/T/ipykernel_32434/2594296392.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv_path)


Original CSV: 18 columns, 180418 rows
New CSV: 19 columns, 180418 rows
New CSV saved at: /Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID.csv


## this filters all the pdf id in the folder with the pdf id column in the csv and gives onyl those rows

In [ ]:

import pandas as pd
import os
import requests
import subprocess
import sys
import json 

# Define file paths for input CSV and PDF
input_csv_path = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID.csv'
sample_questions_csv_path = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/sample_questions_for_template - sample_questions_for_template.csv'
pdf_file_path = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/P01_10021040891039611111693747018.pdf'

# Load the original CSV file
df = pd.read_csv(input_csv_path, low_memory=False)

# Check if the PDF_ID column exists and the data is loaded correctly
if 'PDF_ID' not in df.columns:
    print(f"Error: 'PDF_ID' column not found in the CSV.")
else:
    # Extract PDF ID from the filename (removes 'P01_' and '.pdf')
    pdf_id = os.path.basename(pdf_file_path).replace('P01_', '').replace('.pdf', '')

    # Remove the '.pdf' extension from the PDF_ID column in the CSV
    df['PDF_ID'] = df['PDF_ID'].str.replace('.pdf', '', regex=False)

    # Filter the rows where PDF_ID matches the extracted ID
    filtered_df = df[df['PDF_ID'] == pdf_id]

    # If no rows match, inform the user
    if filtered_df.empty:
        print(f"No rows found for PDF_ID: {pdf_id}")
    else:
        # Select the relevant columns
        columns_to_copy = ['QB_ID', 'textsolutions', 'Marks', 'Question_no', 'oldtags', 'versionid', 'SUBJECT', 'content', 'UPLOADED_ANS', 'oldquestionid' , 'PDF_ID']
        filtered_df = filtered_df[columns_to_copy]

        # Load the sample_questions_for_template CSV
        sample_questions_df = pd.read_csv(sample_questions_csv_path)

        # Ensure 'oldquestionid' is treated as a string before applying str.replace()
        filtered_df['oldquestionid'] = filtered_df['oldquestionid'].astype(str).str.replace('.0', '', regex=False)
        sample_questions_df['oldquestionid'] = sample_questions_df['oldquestionid'].astype(str).str.replace('.0', '', regex=False)

        # Initialize counters for copied and not copied rows
        copied_rows_count = 0
        not_found_count = 0

        # Add columns from sample_questions_df to the filtered_df
        filtered_df['qb_textsolutions'] = None
        filtered_df['brainly_solution'] = None
        filtered_df['allie_solution(allen_bot)'] = None

        # Iterate through each row in the filtered DataFrame
        for index, row in filtered_df.iterrows():
            oldquestionid = row['oldquestionid']
            
            # Check if the oldquestionid exists in sample_questions_df
            match = sample_questions_df[sample_questions_df['oldquestionid'] == oldquestionid]

            if not match.empty:
                # Copy the corresponding values to the filtered_df
                filtered_df.at[index, 'qb_textsolutions'] = match.iloc[0]['qb_textsolutions']
                filtered_df.at[index, 'brainly_solution'] = match.iloc[0]['brainly_solution']
                filtered_df.at[index, 'allie_solution(allen_bot)'] = match.iloc[0]['allie_solution(allen_bot)']
                copied_rows_count += 1
            else:
                not_found_count += 1

        # Define the output file path with the extracted PDF ID
        output_csv_path = f'/Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID_{pdf_id}.csv'

        # Save the filtered DataFrame to a new CSV
        filtered_df.to_csv(output_csv_path, index=False)

        # Print details
        print(f"Filtered CSV saved at: {output_csv_path}")
        print(f"Rows copied from the first CSV: {filtered_df.shape[0]}")
        print(f"Rows with matching oldquestionid found in the second CSV: {copied_rows_count}")
        print(f"Rows with no matching oldquestionid found in the second CSV: {not_found_count}")


# ---------------------------- DOWNLOAD PDFs ---------------------------- #

# Define file paths for PDF download
input_csv_path_2 = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID_10021040891039611111693747018.csv'
output_folder = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/Phy_01'

# Load the CSV file with the filtered data
df = pd.read_csv(input_csv_path_2)

# Ensure the directory exists
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Initialize counters for downloaded and not downloaded PDFs
downloaded_count = 0
not_downloaded_count = 0

# Function to download PDF from the URL
def download_pdf(url, folder_path, pdf_name):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            # Save the PDF with the given name in the specified folder
            file_path = os.path.join(folder_path, pdf_name)
            with open(file_path, 'wb') as f:
                f.write(response.content)
            return file_path  # Return the file path for further processing
        else:
            return None
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return None

# Iterate over each row in the dataframe
downloaded_files = set()  # Track downloaded PDFs

# Iterate over each row in the dataframe
for index, row in df.iterrows():
    url = row['UPLOADED_ANS']
    if isinstance(url, str) and url.startswith('https://'):  # Ensure it's a valid URL
        # Extract the PDF name from the URL (after the last '/')
        pdf_name = url.split('/')[-1]

        # Skip if the PDF has already been downloaded
        if pdf_name in downloaded_files:
            print(f"Skipping download for already downloaded PDF: {pdf_name}")
            continue

        # Create a subfolder for the PDF
        pdf_folder_path = os.path.join(output_folder, pdf_name.split('.')[0])
        if not os.path.exists(pdf_folder_path):
            os.makedirs(pdf_folder_path)

        # Download the PDF
        pdf_file_path = download_pdf(url, pdf_folder_path, pdf_name)

        if pdf_file_path:
            downloaded_files.add(pdf_name)  # Add to set to track downloads
            downloaded_count += 1
            print(f"Downloaded: {pdf_file_path}")
            
            # Run the trial.py script and process the PDF
            try:
                subprocess.run(['python3', trial_script_path, pdf_file_path], check=True, timeout=120)  # Increased timeout
                print(f"Successfully processed PDF: {pdf_file_path}")
            except subprocess.CalledProcessError as e:
                print(f"Error running trial.py for PDF: {pdf_file_path}, {e}")
            except subprocess.TimeoutExpired:
                print(f"Timeout expired while processing PDF: {pdf_file_path}")
        else:
            not_downloaded_count += 1

# Print the results
print(f"Total PDFs downloaded: {downloaded_count}")
print(f"Total PDFs not downloaded: {not_downloaded_count}")

# Ensure script exits once all PDFs are processed
print("All PDFs have been processed. Exiting script.")
# ---------------------------- ADD JSON CONTENT TO CSV ---------------------------- #

# Dynamically construct the JSON file path based on the PDF's name and folder
pdf_name_without_extension = os.path.basename(pdf_file_path).replace('.pdf', '')
json_file_path = os.path.join(pdf_folder_path, "output.json")

# Check if the JSON file exists
if os.path.exists(json_file_path):
    # Load the JSON content
    with open(json_file_path, 'r') as json_file:
        json_data = json.load(json_file)

    # Load the CSV file to update it
    df = pd.read_csv(input_csv_path_2, low_memory=False)

    # Strip any leading or trailing spaces from URLs to ensure uniformity
    df['UPLOADED_ANS'] = df['UPLOADED_ANS'].str.strip()

    # Extract the PDF ID from the URL (the last part of the URL, after the last '/')
    df['extracted_pdf_id'] = df['UPLOADED_ANS'].apply(lambda x: x.split('/')[-1].replace('.pdf', '') if isinstance(x, str) else '')

    # Check if the extracted PDF ID matches the PDF_ID in the CSV
    df['student_gemini_ocr_solution'] = df.apply(
        lambda row: json_data if row['extracted_pdf_id'] == row['PDF_ID'] else None, axis=1
    )

    # Dynamically generate the output CSV path based on input_csv_path_2
    # Extract PDF ID from the input CSV path
    pdf_id = os.path.basename(input_csv_path_2).replace('hw_df_with_solutions_and_questions_PDF_ID_', '').replace('.csv', '')

    # Define the output file path with the extracted PDF ID
    output_csv_path = f'/Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID_{pdf_id}.csv'

    # Save the filtered DataFrame to the correct CSV file
    df.to_csv(output_csv_path, index=False)
    print(f"Added 'student_gemini_ocr_solution' column and updated the CSV file at {output_csv_path}")
else:
    print(f"Error: JSON file not found at {json_file_path}. Skipping this step.")

# Ensure script exits once all PDFs are processed
print("All PDFs have been processed. Exiting script.")
sys.exit(0)  # Exit the script gracefully






Filtered CSV saved at: /Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID_10021040891039611111693747018.csv
Rows copied from the first CSV: 11
Rows with matching oldquestionid found in the second CSV: 0
Rows with no matching oldquestionid found in the second CSV: 11
Downloaded: /Users/simrannaik/Desktop/solution_improvement/HW_DF/Phy_01/10021040891039611111693747018/10021040891039611111693747018.pdf
OCR results saved to /Users/simrannaik/Desktop/solution_improvement/HW_DF/Phy_01/10021040891039611111693747018/output.json
Successfully processed PDF: /Users/simrannaik/Desktop/solution_improvement/HW_DF/Phy_01/10021040891039611111693747018/10021040891039611111693747018.pdf
Skipping download for already downloaded PDF: 10021040891039611111693747018.pdf
Skipping download for already downloaded PDF: 10021040891039611111693747018.pdf
Skipping download for already downloaded PDF: 10021040891039611111693747018.pdf
Skipping download for already download

SystemExit: 0

/Users/simrannaik/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## this takes the gemini student reponse for the uploadede-ans since the ans is same for all the rows

In [ ]:
import pandas as pd
import json

# File paths
csv_file_path = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID_10021040891039611111693747018.csv'
json_file_path = '/Users/simrannaik/Desktop/solution_improvement/HW_DF/P01_10021040891039611111693747018-gemini-2.5-pro_output.json'

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Load the JSON file
with open(json_file_path, 'r') as json_file:
    json_data = json.load(json_file)

# Check if all URLs in the 'UPLOADED_ANS' column are the same
if df['UPLOADED_ANS'].nunique() == 1:  # All rows have the same URL
    # Add a new column 'student_gemini_ocr_solution' and assign the JSON data to it
    df['student_gemini_ocr_solution'] = [json_data] * len(df)  # Repeat the JSON data for all rows

    # Save the updated DataFrame back to the CSV
    df.to_csv(csv_file_path, index=False)
    print(f"Added 'student_gemini_ocr_solution' column and updated the CSV file at {csv_file_path}")
else:
    print("URLs in the 'UPLOADED_ANS' column are not the same in all rows.")


Added 'student_gemini_ocr_solution' column and updated the CSV file at /Users/simrannaik/Desktop/solution_improvement/HW_DF/hw_df_with_solutions_and_questions_PDF_ID_10021040891039611111693747018.csv
